In [1]:
# ============================================================
# STAGE 1 — DOCUMENT CHARACTERISATION AND QUALITY ASSESSMENT
# D7 — UK National Audit Office STEM Report
# ============================================================

!pip install PyMuPDF
from google.colab import files
from pathlib import Path

import hashlib
import json
import platform
import re
import sys

import fitz
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 55.5 MB/s eta 0:00:00


In [2]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D7"

DOCUMENT_NAME = (
    "UK National Audit Office — Delivering STEM "
    "(science, technology, engineering and mathematics) "
    "skills for the economy"
)

SOURCE_FORMAT = "PDF"
EXPECTED_PAGE_COUNT = 12
EXPECTED_REFERENCE_RECORD_COUNT = 59

EXPECTED_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

EXPECTED_CATEGORY_COUNTS = {
    "Key fact": 10,
    "Policy context": 3,
    "Policy finding": 7,
    "Education pipeline statistic": 24,
    "Government initiative": 9,
    "Recommendation": 6
}

OUTPUT_DIR = Path("outputs_D7_stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_VALUES_PATH = (
    OUTPUT_DIR / "D7_reference_values.csv"
)

REFERENCE_VALUES_JSON_PATH = (
    OUTPUT_DIR / "D7_reference_values.json"
)

REFERENCE_SCHEMA_PATH = (
    OUTPUT_DIR / "D7_reference_schema.json"
)

REFERENCE_SUMMARY_PATH = (
    OUTPUT_DIR / "D7_reference_summary.json"
)

DOCUMENT_CHARACTERISATION_PATH = (
    OUTPUT_DIR / "D7_document_characterisation.json"
)

PAGE_CHARACTERISATION_PATH = (
    OUTPUT_DIR / "D7_page_characterisation.csv"
)

QUALITY_EVIDENCE_PATH = (
    OUTPUT_DIR / "D7_quality_evidence.json"
)

EXTRACTION_SCHEMA_PATH = (
    OUTPUT_DIR / "D7_extraction_schema.json"
)

EXTRACTION_TASK_PATH = (
    OUTPUT_DIR / "D7_extraction_task.txt"
)

INDICATOR_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D7_indicator_assessment.csv"
)

DIMENSION_ASSESSMENT_PATH = (
    OUTPUT_DIR / "D7_dimension_assessment.csv"
)

REFERENCE_METADATA_PATH = (
    OUTPUT_DIR / "D7_reference_metadata.json"
)

REFERENCE_INTEGRITY_PATH = (
    OUTPUT_DIR / "D7_reference_integrity.json"
)

print("Document:", DOCUMENT_ID)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print("Expected reference records:", EXPECTED_REFERENCE_RECORD_COUNT)


Document: D7
Expected pages: 12
Expected reference records: 59


In [3]:
# ============================================================
# 2. Upload source PDF
# ============================================================

print(
    "Upload the D7 UK National Audit Office STEM Report PDF."
)

uploaded = files.upload()

pdf_paths = [
    Path(filename)
    for filename in uploaded
    if filename.lower().endswith(".pdf")
]

if len(pdf_paths) != 1:
    raise ValueError(
        "Upload exactly one PDF source document."
    )

SOURCE_PATH = pdf_paths[0]

print("Source file:", SOURCE_PATH.name)
print("Source size:", f"{SOURCE_PATH.stat().st_size:,} bytes")


Upload the D7 UK National Audit Office STEM Report PDF.


Saving D7 - UK National Audit Office – STEM Report.pdf to D7 - UK National Audit Office – STEM Report.pdf
Source file: D7 - UK National Audit Office – STEM Report.pdf
Source size: 125,339 bytes


In [4]:
# ============================================================
# 3. File hashing utility
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(SOURCE_PATH)

print("Source SHA-256:", SOURCE_SHA256)


Source SHA-256: 00cd2555312b220d7b4289144261aaba888336fb21b32b9ff53e96427a5f7eba


In [5]:
# ============================================================
# 4. Load and inspect PDF
# ============================================================

pdf_document = fitz.open(SOURCE_PATH)

PAGE_COUNT = len(pdf_document)
PAGE_COUNT_VALID = (
    PAGE_COUNT == EXPECTED_PAGE_COUNT
)

page_rows = []
page_texts = []

for page_number, page in enumerate(
    pdf_document,
    start=1
):
    text = page.get_text("text") or ""

    page_texts.append(text)

    page_rows.append(
        {
            "Page Number": page_number,
            "Character Count": len(text),
            "Word Count": len(text.split()),
            "Numeric Token Count": len(
                re.findall(
                    r"(?<!\w)[£$]?\(?-?\d[\d,]*(?:\.\d+)?%?\)?",
                    text
                )
            ),
            "Text Extractable": bool(text.strip())
        }
    )


page_characterisation_df = pd.DataFrame(
    page_rows
)

FULL_TEXT = "\n".join(page_texts)

TEXT_EXTRACTABLE = bool(
    page_characterisation_df[
        "Text Extractable"
    ].all()
)

OCR_REQUIRED = not TEXT_EXTRACTABLE

print("Page count:", PAGE_COUNT)
print("Page count valid:", PAGE_COUNT_VALID)
print("Text extractable:", TEXT_EXTRACTABLE)
print("OCR required:", OCR_REQUIRED)

display(page_characterisation_df)

if not PAGE_COUNT_VALID:
    raise AssertionError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, found {PAGE_COUNT}."
    )

if not TEXT_EXTRACTABLE:
    raise AssertionError(
        "The D7 source PDF does not contain extractable text."
    )


Page count: 12
Page count valid: True
Text extractable: True
OCR required: False


,Page Number,Character Count,Word Count,Numeric Token Count,Text Extractable
0,1,307,43,5,True
1,2,1204,184,2,True
2,3,555,88,9,True
3,4,1608,239,4,True
4,5,1117,168,15,True
5,6,1112,166,29,True
6,7,2486,368,7,True
7,8,2418,371,16,True
8,9,3560,562,21,True
9,10,2975,453,69,True


In [6]:
# ============================================================
# 5. Document characterisation
# ============================================================

numeric_tokens = re.findall(
    r"(?<!\w)[£$]?\(?-?\d[\d,]*(?:\.\d+)?%?\)?",
    FULL_TEXT
)

word_count = len(
    FULL_TEXT.split()
)

numeric_token_to_word_ratio = (
    len(numeric_tokens) / word_count
    if word_count
    else 0
)


DOCUMENT_CHARACTERISATION = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "source_format": SOURCE_FORMAT,
    "page_count": PAGE_COUNT,
    "page_count_valid": PAGE_COUNT_VALID,
    "text_extractable": TEXT_EXTRACTABLE,
    "ocr_required": OCR_REQUIRED,
    "total_character_count": len(FULL_TEXT),
    "total_word_count": len(FULL_TEXT.split()),

    "numeric_token_count":
        len(numeric_tokens),

    "numeric_token_to_word_ratio":
        round(
            numeric_token_to_word_ratio,
            3
        ),

    "money_token_count": len(
        re.findall(
            r"£\s?\d[\d,]*(?:\.\d+)?",
            FULL_TEXT
        )
    ),

    "percentage_token_count": len(
        re.findall(
            r"\d+(?:\.\d+)?\s?%",
            FULL_TEXT
        )
    ),

    "represented_sections": [
        "Title and publication material",
        "Contents",
        "Key facts",
        "Summary — Background",
        "Summary — Government intervention",
        "Summary — Scope and approach",
        "Summary — Key findings",
        "Summary — Conclusion on value for money",
        "Summary — Recommendations"
    ],

    "fixed_extraction_scope": (
        "Quantitative key facts and summary statistics; "
        "headline policy findings; government initiatives; "
        "value-for-money conclusion; and recommendations "
        "represented on pages 6–12 of the supplied PDF."
    ),

    "excluded_from_reference_scope": [
        "Publication metadata",
        "Copyright and contact information",
        "Contents-page entries",
        "General descriptive narrative without a fixed finding, "
        "initiative, recommendation or requested quantitative record",
        "Values appearing only as publication or contact metadata"
    ],

}


print(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D7",
  "document_name": "UK National Audit Office — Delivering STEM (science, technology, engineering and mathematics) skills for the economy",
  "source_file": "D7 - UK National Audit Office – STEM Report.pdf",
  "source_file_sha256": "00cd2555312b220d7b4289144261aaba888336fb21b32b9ff53e96427a5f7eba",
  "source_format": "PDF",
  "page_count": 12,
  "page_count_valid": true,
  "text_extractable": true,
  "ocr_required": false,
  "total_character_count": 22823,
  "total_word_count": 3473,
  "numeric_token_count": 213,
  "numeric_token_to_word_ratio": 0.061,
  "money_token_count": 6,
  "percentage_token_count": 21,
  "represented_sections": [
    "Title and publication material",
    "Contents",
    "Key facts",
    "Summary — Background",
    "Summary — Government intervention",
    "Summary — Scope and approach",
    "Summary — Key findings",
    "Summary — Conclusion on value for money",
    "Summary — Recommendations"
  ],
  "fixed_extraction_scope": "Quantitative

In [7]:
# ============================================================
# 6. Definition of fixed extraction task
# ============================================================

EXTRACTION_TASK = """
Extract the fixed subset of policy and quantitative records represented
in the supplied D7 National Audit Office STEM report.

Use only the supplied document as evidence.

Include:

- the 10 Key Facts records;
- the predefined policy-context records;
- the predefined headline policy findings;
- the predefined education-pipeline statistics;
- the predefined government initiatives;
- the six recommendations.

For every record return:

- Category
- Statement or Section
- Metric
- Topic
- Value
- Unit
- Qualifier
- Reporting Period
- Source Location

Rules:

- Extract only information explicitly represented in the source.
- Preserve repeated statistics when they occur in different source
  sections.
- Preserve source measurement scale.
- Preserve approximation or inequality wording such as around,
  almost, over and more than in the Qualifier field.
- Use null for Value, Unit, Qualifier or Reporting Period when the
  corresponding information is not explicitly represented.
- Do not calculate, infer, derive, convert, repair or deduplicate
  source values or introduce information not explicitly represented
  in the source.
- Do not extract publication metadata, copyright information,
  contact details or contents-page entries.
- Return exactly 59 records.
- Return valid JSON using the exact field names defined in the
  extraction schema.
- Do not include explanations before or after the JSON.
"""

print(EXTRACTION_TASK)


Extract the fixed subset of policy and quantitative records represented
in the supplied D7 National Audit Office STEM report.

Use only the supplied document as evidence.

Include:

- the 10 Key Facts records;
- the predefined policy-context records;
- the predefined headline policy findings;
- the predefined education-pipeline statistics;
- the predefined government initiatives;
- the six recommendations.

For every record return:

- Category
- Statement or Section
- Metric
- Topic
- Value
- Unit
- Qualifier
- Reporting Period
- Source Location

Rules:

- Extract only information explicitly represented in the source.
- Preserve repeated statistics when they occur in different source
  sections.
- Preserve source measurement scale.
- Preserve approximation or inequality wording such as around,
  almost, over and more than in the Qualifier field.
- Use null for Value, Unit, Qualifier or Reporting Period when the
  corresponding information is not explicitly represented.
- Do not calcul

In [8]:
# ============================================================
# 7. Extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,

    "record_level":
        "policy_or_quantitative_summary_record",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category": {
            "type": ["string", "null"]
        },
        "Statement or Section": {
            "type": ["string", "null"]
        },
        "Metric": {
            "type": ["string", "null"]
        },
        "Topic": {
            "type": ["string", "null"]
        },
        "Value": {
            "type": ["number", "null"]
        },
        "Unit": {
            "type": ["string", "null"]
        },
        "Qualifier": {
            "type": ["string", "null"]
        },
        "Reporting Period": {
            "type": ["string", "null"]
        },
        "Source Location": {
            "type": ["string", "null"]
        }
    },

    "expected_output_structure": {
        "document_id": DOCUMENT_ID,
        "records": [
            {
                "Category": "string or null",
                "Statement or Section": "string or null",
                "Metric": "string or null",
                "Topic": "string or null",
                "Value": "number or null",
                "Unit": "string or null",
                "Qualifier": "string or null",
                "Reporting Period": "string or null",
                "Source Location": "string or null"
            }
        ]
    }
}

print(
    json.dumps(
        EXTRACTION_SCHEMA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D7",
  "record_level": "policy_or_quantitative_summary_record",
  "expected_record_count": 59,
  "fields": {
    "Category": {
      "type": [
        "string",
        "null"
      ]
    },
    "Statement or Section": {
      "type": [
        "string",
        "null"
      ]
    },
    "Metric": {
      "type": [
        "string",
        "null"
      ]
    },
    "Topic": {
      "type": [
        "string",
        "null"
      ]
    },
    "Value": {
      "type": [
        "number",
        "null"
      ]
    },
    "Unit": {
      "type": [
        "string",
        "null"
      ]
    },
    "Qualifier": {
      "type": [
        "string",
        "null"
      ]
    },
    "Reporting Period": {
      "type": [
        "string",
        "null"
      ]
    },
    "Source Location": {
      "type": [
        "string",
        "null"
      ]
    }
  },
  "expected_output_structure": {
    "document_id": "D7",
    "records": [
      {
        "Category": "string or

In [9]:
# ============================================================
# 8. Reference schema
# ============================================================

REFERENCE_SCHEMA = {
    "document_id": DOCUMENT_ID,

    "record_level":
        "policy_or_quantitative_summary_record",

    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "fields": {
        "Category":
            "Fixed reference-record category",

        "Statement or Section":
            "Named source section, paragraph or recommendation",

        "Metric":
            "Source-grounded quantitative observation, finding, "
            "initiative or recommendation",

        "Topic":
            "Main subject associated with the record",

        "Value":
            "Explicit numeric value where represented in the source",

        "Unit":
            "Measurement unit associated with Value",

        "Qualifier":
            "Explicit approximation or inequality wording associated "
            "with Value, such as around, almost, over, more than or minimum",

        "Reporting Period":
            "Explicit reporting period or temporal reference where represented",

        "Source Location":
            "PDF page and source section, paragraph or recommendation"
    },

    "null_policy": (
        "Value, Unit, Qualifier and Reporting Period are null when the "
        "corresponding information is not explicitly represented in the source."
    ),

    "branch_reuse": (
        "The same fixed reference dataset is reused for "
        "Branches A, B and C."
    )
}

print(
    json.dumps(
        REFERENCE_SCHEMA,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_id": "D7",
  "record_level": "policy_or_quantitative_summary_record",
  "expected_record_count": 59,
  "fields": {
    "Category": "Fixed reference-record category",
    "Statement or Section": "Named source section, paragraph or recommendation",
    "Metric": "Source-grounded quantitative observation, finding, initiative or recommendation",
    "Topic": "Main subject associated with the record",
    "Value": "Explicit numeric value where represented in the source",
    "Unit": "Measurement unit associated with Value",
    "Qualifier": "Explicit approximation or inequality wording associated with Value, such as around, almost, over, more than or minimum",
    "Reporting Period": "Explicit reporting period or temporal reference where represented",
    "Source Location": "PDF page and source section, paragraph or recommendation"
  },
  "null_policy": "Value, Unit, Qualifier and Reporting Period are null when the corresponding information is not explicitly represented in the

In [10]:
# ============================================================
# 9. Fixed D7 reference records
# ============================================================

reference_records = [
    # --------------------------------------------------------
    # Key facts — page 6
    # --------------------------------------------------------
    {
        "Category": "Key fact",
        "Statement or Section": "Key facts",
        "Metric": "Spent on, or committed to, key STEM-specific interventions",
        "Topic": "STEM interventions",
        "Value": 990,
        "Unit": "GBP million",
        "Reporting Period": "2007 to autumn 2017",
        "Source Location": "PDF page 6 — Key facts"
    },
    {
        "Category": "Key fact",
        "Statement or Section": "Key facts",
        "Metric": "Undergraduate enrolments in STEM subjects",
        "Topic": "Undergraduate STEM participation",
        "Value": 442000,
        "Unit": "enrolments",
        "Reporting Period": "2015/16",
        "Source Location": "PDF page 6 — Key facts"
    },
    {
        "Category": "Key fact",
        "Statement or Section": "Key facts",
        "Metric": "Graduates in STEM subjects known to be working in a STEM occupation six months later",
        "Topic": "STEM graduate destinations",
        "Value": 24,
        "Unit": "percent",
        "Reporting Period": "Six months after graduation",
        "Source Location": "PDF page 6 — Key facts"
    },
    {
        "Category": "Key fact",
        "Statement or Section": "Key facts",
        "Metric": "Additional STEM technicians estimated as needed to meet employer demand",
        "Topic": "STEM technician demand",
        "Value": 700000,
        "Unit": "technicians",
        "Reporting Period": "Decade to 2024",
        "Source Location": "PDF page 6 — Key facts"
    },
    {
        "Category": "Key fact",
        "Statement or Section": "Key facts",
        "Metric": "STEM apprenticeship starts",
        "Topic": "STEM apprenticeships",
        "Value": 112000,
        "Unit": "starts",
        "Reporting Period": "2016/17",
        "Source Location": "PDF page 6 — Key facts"
    },
    {
        "Category": "Key fact",
        "Statement or Section": "Key facts",
        "Metric": "STEM apprenticeships started by women",
        "Topic": "Female participation in STEM apprenticeships",
        "Value": 8,
        "Unit": "percent",
        "Qualifier": None,
        "Reporting Period": "2016/17",
        "Source Location": "PDF page 6 — Key facts"
    },
    {
        "Category": "Key fact",
        "Statement or Section": "Key facts",
        "Metric": "Government investment in national colleges",
        "Topic": "National colleges",
        "Value": 80,
        "Unit": "GBP million",
        "Reporting Period": None,
        "Source Location": "PDF page 6 — Key facts"
    },
    {
        "Category": "Key fact",
        "Statement or Section": "Key facts",
        "Metric": "Rise in STEM A level examination entries compared with the previous year",
        "Topic": "STEM A level entries",
        "Value": 2.6,
        "Unit": "percent",
        "Reporting Period": "2016/17",
        "Source Location": "PDF page 6 — Key facts"
    },
    {
        "Category": "Key fact",
        "Statement or Section": "Key facts",
        "Metric": "Fall in enrolments in part-time undergraduate STEM degrees",
        "Topic": "Part-time undergraduate STEM participation",
        "Value": -30.9,
        "Unit": "percent",
        "Reporting Period": "2011/12 to 2015/16",
        "Source Location": "PDF page 6 — Key facts"
    },
    {
        "Category": "Key fact",
        "Statement or Section": "Key facts",
        "Metric": "Government capital investment in higher education STEM provision",
        "Topic": "Higher education STEM provision",
        "Value": 200,
        "Unit": "GBP million",
        "Reporting Period": "2015/16",
        "Source Location": "PDF page 6 — Key facts"
    },

    # --------------------------------------------------------
    # Policy context — page 7
    # --------------------------------------------------------
    {
        "Category": "Policy context",
        "Statement or Section": "Summary paragraph 1 — Background",
        "Metric": "There is no universally accepted definition of STEM in either education or employment",
        "Topic": "Definition of STEM",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 7 — Summary paragraph 1"
    },
    {
        "Category": "Policy context",
        "Statement or Section": "Summary paragraph 3 — Background",
        "Metric": "The key routes for developing STEM knowledge and skills are schools and sixth-form colleges, further education colleges, apprenticeships and higher education institutions",
        "Topic": "STEM skills pipeline",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 7 — Summary paragraph 3"
    },
    {
        "Category": "Policy context",
        "Statement or Section": "Summary paragraph 4 — Government intervention",
        "Metric": "DfE is responsible for the majority of STEM skills interventions and BEIS has a cross-cutting role",
        "Topic": "Departmental responsibility",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 7 — Summary paragraph 4"
    },

    # --------------------------------------------------------
    # Policy findings and conclusion — pages 8–11
    # --------------------------------------------------------
    {
        "Category": "Policy finding",
        "Statement or Section": "Summary paragraph 7 — Key findings",
        "Metric": "Government does not currently gather robust intelligence on the STEM skills issues it has already started to address",
        "Topic": "Labour market intelligence",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 8 — Summary paragraph 7"
    },
    {
        "Category": "Policy finding",
        "Statement or Section": "Summary paragraph 8 — Key findings",
        "Metric": "Current estimates of the STEM skills problem vary widely and typically focus only on individual sections of the workforce",
        "Topic": "Estimates of STEM skills needs",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 9 — Summary paragraph 8"
    },
    {
        "Category": "Policy finding",
        "Statement or Section": "Summary paragraph 9 — Key findings",
        "Metric": "Government does not have a stable and consistent set of definitions for STEM in either an educational or a work context",
        "Topic": "STEM definitions",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 9 — Summary paragraph 9"
    },
    {
        "Category": "Policy finding",
        "Statement or Section": "Summary paragraph 10 — Key findings",
        "Metric": "Existing evidence indicates that there is a STEM skills mismatch rather than a simple shortage",
        "Topic": "STEM skills mismatch",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 9 — Summary paragraph 10"
    },
    {
        "Category": "Policy finding",
        "Statement or Section": "Summary paragraph 11 — Key findings",
        "Metric": "Government is starting to improve coordination on STEM and address past incoherence",
        "Topic": "Cross-government coordination",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 9 — Summary paragraph 11"
    },
    {
        "Category": "Policy finding",
        "Statement or Section": "Summary paragraph 12 — Key findings",
        "Metric": "The impact of exit from the EU is difficult to predict",
        "Topic": "EU exit and STEM skills",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 9 — Summary paragraph 12"
    },
    {
        "Category": "Policy finding",
        "Statement or Section": "Summary paragraph 21 — Conclusion on value for money",
        "Metric": "The absence of a precise understanding of the STEM skills problem means the efforts of DfE and BEIS are not well prioritised and a better targeted approach is needed to demonstrate value for money",
        "Topic": "Value for money",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 11 — Summary paragraph 21"
    },

    # --------------------------------------------------------
    # Education pipeline statistics — pages 10–11
    # Repeated figures are retained where represented again.
    # --------------------------------------------------------
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 13 — Education pipeline",
        "Metric": "Growth in A level entries targeted by STEM initiatives",
        "Topic": "STEM A level initiatives",
        "Value": 3,
        "Unit": "percent",
        "Reporting Period": "2011/12 to 2016/17",
        "Source Location": "PDF page 10 — Summary paragraph 13"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 14 — Education pipeline",
        "Metric": "Female students as a share of all STEM A level examination entries",
        "Topic": "Female participation in STEM A levels",
        "Value": 42,
        "Unit": "percent",
        "Reporting Period": "2016/17",
        "Source Location": "PDF page 10 — Summary paragraph 14"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 14 — Education pipeline",
        "Metric": "Female students as a share of computing examination entries",
        "Topic": "Female participation in computing",
        "Value": 9.4,
        "Unit": "percent",
        "Reporting Period": "2016/17",
        "Source Location": "PDF page 10 — Summary paragraph 14"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 14 — Education pipeline",
        "Metric": "Female students as a share of physics examination entries",
        "Topic": "Female participation in physics",
        "Value": 21.2,
        "Unit": "percent",
        "Reporting Period": "2016/17",
        "Source Location": "PDF page 10 — Summary paragraph 14"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 14 — Education pipeline",
        "Metric": "Female students as a share of mathematics examination entries",
        "Topic": "Female participation in mathematics",
        "Value": 39,
        "Unit": "percent",
        "Reporting Period": "2016/17",
        "Source Location": "PDF page 10 — Summary paragraph 14"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 14 — Education pipeline",
        "Metric": "Female students as a share of STEM apprenticeship starts",
        "Topic": "Female participation in STEM apprenticeships",
        "Value": 8,
        "Unit": "percent",
        "Qualifier": "around",
        "Reporting Period": "2016/17",
        "Source Location": "PDF page 10 — Summary paragraph 14"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 14 — Education pipeline",
        "Metric": "Women as a share of all apprenticeship starts",
        "Topic": "Female participation in all apprenticeships",
        "Value": 50,
        "Unit": "percent",
        "Qualifier": "more than",
        "Reporting Period": "2016/17",
        "Source Location": "PDF page 10 — Summary paragraph 14"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 14 — Education pipeline",
        "Metric": "Female students as a share of undergraduate STEM enrolments",
        "Topic": "Female participation in undergraduate STEM",
        "Value": 38,
        "Unit": "percent",
        "Qualifier": "around",
        "Reporting Period": None,
        "Source Location": "PDF page 10 — Summary paragraph 14"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 14 — Education pipeline",
        "Metric": "Women as a share of all undergraduate enrolments",
        "Topic": "Female participation in all undergraduate courses",
        "Value": 50,
        "Unit": "percent",
        "Qualifier": "more than",
        "Reporting Period": None,
        "Source Location": "PDF page 10 — Summary paragraph 14"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 15 — Education pipeline",
        "Metric": "STEM apprenticeship starts",
        "Topic": "STEM apprenticeships",
        "Value": 95000,
        "Unit": "starts",
        "Reporting Period": "2012/13",
        "Source Location": "PDF page 10 — Summary paragraph 15"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 15 — Education pipeline",
        "Metric": "STEM apprenticeship starts",
        "Topic": "STEM apprenticeships",
        "Value": 112000,
        "Unit": "starts",
        "Reporting Period": "2016/17",
        "Source Location": "PDF page 10 — Summary paragraph 15"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 15 — Education pipeline",
        "Metric": "Non-apprenticeship STEM further education learning aims",
        "Topic": "Further education STEM participation",
        "Value": 110000,
        "Unit": "learning aims",
        "Qualifier": "around",
        "Reporting Period": "2011/12 to 2015/16",
        "Source Location": "PDF page 10 — Summary paragraph 15"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 16 — Education pipeline",
        "Metric": "Rise in full-time STEM degree enrolments",
        "Topic": "Full-time undergraduate STEM participation",
        "Value": 6.9,
        "Unit": "percent",
        "Reporting Period": "2011/12 to 2015/16",
        "Source Location": "PDF page 10 — Summary paragraph 16"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 16 — Education pipeline",
        "Metric": "Rise in enrolments across all subjects",
        "Topic": "All-subject undergraduate participation",
        "Value": 1.1,
        "Unit": "percent",
        "Reporting Period": "2011/12 to 2015/16",
        "Source Location": "PDF page 10 — Summary paragraph 16"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 16 — Education pipeline",
        "Metric": "Fall in part-time undergraduate STEM course take-up",
        "Topic": "Part-time undergraduate STEM participation",
        "Value": -30,
        "Unit": "percent",
        "Qualifier": "over",
        "Reporting Period": "2011/12 to 2015/16",
        "Source Location": "PDF page 10 — Summary paragraph 16"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 16 — Education pipeline",
        "Metric": "Part-time undergraduate STEM course enrolments",
        "Topic": "Part-time undergraduate STEM participation",
        "Value": 98000,
        "Unit": "enrolments",
        "Qualifier": "almost",
        "Reporting Period": "2011/12",
        "Source Location": "PDF page 10 — Summary paragraph 16"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 16 — Education pipeline",
        "Metric": "Part-time undergraduate STEM course enrolments",
        "Topic": "Part-time undergraduate STEM participation",
        "Value": 68000,
        "Unit": "enrolments",
        "Qualifier": "around",
        "Reporting Period": "2015/16",
        "Source Location": "PDF page 10 — Summary paragraph 16"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 16 — Education pipeline",
        "Metric": "Overall fall in part-time degree enrolments",
        "Topic": "Part-time undergraduate participation",
        "Value": -47,
        "Unit": "percent",
        "Reporting Period": "2011/12 to 2015/16",
        "Source Location": "PDF page 10 — Summary paragraph 16"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 17 — Education pipeline",
        "Metric": "People who graduated with a STEM degree",
        "Topic": "STEM graduates",
        "Value": 75000,
        "Unit": "graduates",
        "Reporting Period": "2016",
        "Source Location": "PDF page 11 — Summary paragraph 17"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 17 — Education pipeline",
        "Metric": "STEM graduates known to be working in a STEM occupation within six months",
        "Topic": "STEM graduate destinations",
        "Value": 24,
        "Unit": "percent",
        "Reporting Period": "Within six months of graduation",
        "Source Location": "PDF page 11 — Summary paragraph 17"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 17 — Education pipeline",
        "Metric": "STEM graduates whose destinations were unknown",
        "Topic": "STEM graduate destinations",
        "Value": 15000,
        "Unit": "graduates",
        "Reporting Period": "2016",
        "Source Location": "PDF page 11 — Summary paragraph 17"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 17 — Education pipeline",
        "Metric": "STEM graduates whose destinations were unknown",
        "Topic": "STEM graduate destinations",
        "Value": 19.9,
        "Unit": "percent",
        "Reporting Period": "2016",
        "Source Location": "PDF page 11 — Summary paragraph 17"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 17 — Education pipeline",
        "Metric": "STEM graduates going on to further study",
        "Topic": "STEM graduate destinations",
        "Value": 13000,
        "Unit": "graduates",
        "Reporting Period": "2016",
        "Source Location": "PDF page 11 — Summary paragraph 17"
    },
    {
        "Category": "Education pipeline statistic",
        "Statement or Section": "Summary paragraph 17 — Education pipeline",
        "Metric": "STEM graduates going on to further study",
        "Topic": "STEM graduate destinations",
        "Value": 17.6,
        "Unit": "percent",
        "Reporting Period": "2016",
        "Source Location": "PDF page 11 — Summary paragraph 17"
    },

    # --------------------------------------------------------
    # Government initiatives — pages 11
    # --------------------------------------------------------
    {
        "Category": "Government initiative",
        "Statement or Section": "Summary paragraph 18 — Latest initiatives",
        "Metric": "T levels are designed to improve vocational education by standardising qualifications, aligning syllabuses with employer demand and establishing clear routes into careers",
        "Topic": "T levels",
        "Value": 15,
        "Unit": "career routes",
        "Reporting Period": None,
        "Source Location": "PDF page 11 — Summary paragraph 18"
    },
    {
        "Category": "Government initiative",
        "Statement or Section": "Summary paragraph 18 — Latest initiatives",
        "Metric": "National colleges focusing on STEM skills",
        "Topic": "National colleges",
        "Value": 4,
        "Unit": "colleges",
        "Reporting Period": None,
        "Source Location": "PDF page 11 — Summary paragraph 18"
    },
    {
        "Category": "Government initiative",
        "Statement or Section": "Summary paragraph 19 — Latest initiatives",
        "Metric": "Institutes of technology will target skills gaps at levels 4 and upwards, particularly in STEM areas",
        "Topic": "Institutes of technology",
        "Value": 4,
        "Unit": "qualification level",
        "Qualifier": "minimum",
        "Reporting Period": "November 2017 proposal",
        "Source Location": "PDF page 11 — Summary paragraph 19"
    },
    {
        "Category": "Government initiative",
        "Statement or Section": "Summary paragraph 20 — Latest initiatives",
        "Metric": "Maths and physics teacher supply package",
        "Topic": "Teacher supply",
        "Value": 67,
        "Unit": "GBP million",
        "Reporting Period": None,
        "Source Location": "PDF page 11 — Summary paragraph 20"
    },
    {
        "Category": "Government initiative",
        "Statement or Section": "Summary paragraph 20 — Latest initiatives",
        "Metric": "Target for recruiting additional maths and physics teachers",
        "Topic": "Teacher recruitment",
        "Value": 2500,
        "Unit": "teachers",
        "Reporting Period": None,
        "Source Location": "PDF page 11 — Summary paragraph 20"
    },
    {
        "Category": "Government initiative",
        "Statement or Section": "Summary paragraph 20 — Latest initiatives",
        "Metric": "Target for improving the skills of non-specialist teachers",
        "Topic": "Teacher development",
        "Value": 15000,
        "Unit": "teachers",
        "Reporting Period": None,
        "Source Location": "PDF page 11 — Summary paragraph 20"
    },
    {
        "Category": "Government initiative",
        "Statement or Section": "Summary paragraph 20 — Latest initiatives",
        "Metric": "Returning teachers recruited by the return to teaching pilot",
        "Topic": "Return to teaching pilot",
        "Value": 428,
        "Unit": "teachers",
        "Reporting Period": None,
        "Source Location": "PDF page 11 — Summary paragraph 20"
    },
    {
        "Category": "Government initiative",
        "Statement or Section": "Summary paragraph 20 — Latest initiatives",
        "Metric": "Target for returning teachers recruited by the return to teaching pilot",
        "Topic": "Return to teaching pilot",
        "Value": 810,
        "Unit": "teachers",
        "Reporting Period": None,
        "Source Location": "PDF page 11 — Summary paragraph 20"
    },
    {
        "Category": "Government initiative",
        "Statement or Section": "Summary paragraph 20 — Latest initiatives",
        "Metric": "Returning teachers who completed the training provided",
        "Topic": "Return to teaching pilot",
        "Value": 330,
        "Unit": "teachers",
        "Reporting Period": None,
        "Source Location": "PDF page 11 — Summary paragraph 20"
    },

    # --------------------------------------------------------
    # Recommendations — page 12
    # --------------------------------------------------------
    {
        "Category": "Recommendation",
        "Statement or Section": "Recommendation 22(a) — DfE",
        "Metric": "Configure the labour market intelligence generated by Skills Advisory Panels and other mechanisms so that it enables effective decision-making",
        "Topic": "Labour market intelligence",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 12 — Recommendation 22(a)"
    },
    {
        "Category": "Recommendation",
        "Statement or Section": "Recommendation 22(b) — DfE",
        "Metric": "Provide departments with clarity on the different STEM definitions used in different contexts, and reasons for these different definitions",
        "Topic": "STEM definitions",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 12 — Recommendation 22(b)"
    },
    {
        "Category": "Recommendation",
        "Statement or Section": "Recommendation 23(c) — BEIS",
        "Metric": "Strengthen its work to evaluate and identify what is effective in its activities to promote participation in STEM education and skills development, and ensure this is shared with its delivery partners",
        "Topic": "Evaluation and knowledge sharing",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 12 — Recommendation 23(c)"
    },
    {
        "Category": "Recommendation",
        "Statement or Section": "Recommendation 23(d) — BEIS",
        "Metric": "Working with other departments, use data on skills mismatches resulting from EU exit to establish the position across relevant sectors and determine whether key capabilities are at risk",
        "Topic": "EU exit and STEM skills",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 12 — Recommendation 23(d)"
    },
    {
        "Category": "Recommendation",
        "Statement or Section": "Recommendation 24(e) — DfE and other key departments",
        "Metric": "Take steps to influence the skills marketplace in priority areas where insufficient development of STEM skills is taking place",
        "Topic": "Skills marketplace",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 12 — Recommendation 24(e)"
    },
    {
        "Category": "Recommendation",
        "Statement or Section": "Recommendation 24(f) — DfE and other key departments",
        "Metric": "Fully embed a more structured approach to STEM across government",
        "Topic": "Cross-government coordination",
        "Value": None,
        "Unit": None,
        "Reporting Period": None,
        "Source Location": "PDF page 12 — Recommendation 24(f)"
    }
]


reference_values_df = pd.DataFrame(
    reference_records,
    columns=EXPECTED_FIELDS
)

reference_values_df = reference_values_df.astype(object).where(
    pd.notna(reference_values_df),
    None
)

reference_records = reference_values_df.to_dict(
    orient="records"
)

print("Reference records:", len(reference_values_df))

display(reference_values_df.head(15))


Reference records: 59


,Category,Statement or Section,Metric,Topic,Value,Unit,Qualifier,Reporting Period,Source Location
0,Key fact,Key facts,"Spent on, or committed to, key STEM-specific i...",STEM interventions,990.0,GBP million,None,2007 to autumn 2017,PDF page 6 — Key facts
1,Key fact,Key facts,Undergraduate enrolments in STEM subjects,Undergraduate STEM participation,442000.0,enrolments,None,2015/16,PDF page 6 — Key facts
2,Key fact,Key facts,Graduates in STEM subjects known to be working...,STEM graduate destinations,24.0,percent,None,Six months after graduation,PDF page 6 — Key facts
3,Key fact,Key facts,Additional STEM technicians estimated as neede...,STEM technician demand,700000.0,technicians,None,Decade to 2024,PDF page 6 — Key facts
4,Key fact,Key facts,STEM apprenticeship starts,STEM apprenticeships,112000.0,starts,None,2016/17,PDF page 6 — Key facts
5,Key fact,Key facts,STEM apprenticeships started by women,Female participation in STEM apprenticeships,8.0,percent,None,2016/17,PDF page 6 — Key facts
6,Key fact,Key facts,Government investment in national colleges,National colleges,80.0,GBP million,None,None,PDF page 6 — Key facts
7,Key fact,Key facts,Rise in STEM A level examination entries compa...,STEM A level entries,2.6,percent,None,2016/17,PDF page 6 — Key facts
8,Key fact,Key facts,Fall in enrolments in part-time undergraduate ...,Part-time undergraduate STEM participation,-30.9,percent,None,2011/12 to 2015/16,PDF page 6 — Key facts
9,Key fact,Key facts,Government capital investment in higher educat...,Higher education STEM provision,200.0,GBP million,None,2015/16,PDF page 6 — Key facts


In [11]:
# ============================================================
# 10. Validation of reference schema and field types
# ============================================================

reference_schema_valid = (
    reference_values_df.columns.tolist()
    == EXPECTED_FIELDS
)

record_count_valid = (
    len(reference_values_df)
    == EXPECTED_REFERENCE_RECORD_COUNT
)

observed_category_counts = (
    reference_values_df["Category"]
    .value_counts()
    .to_dict()
)

category_counts_valid = (
    observed_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

schema_issue_rows = []
type_issue_rows = []
missing_mandatory_rows = []

mandatory_string_fields = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Source Location"
]

nullable_string_fields = [
    "Unit",
    "Qualifier",
    "Reporting Period"
]

for row_index, row in reference_values_df.iterrows():

    for field in mandatory_string_fields:
        value = row[field]

        if value is None or value == "":
            missing_mandatory_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field
                }
            )

        elif not isinstance(value, str):
            type_issue_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field,
                    "Observed Type": type(value).__name__
                }
            )

    for field in nullable_string_fields:
        value = row[field]

        if (
            value is not None
            and not isinstance(value, str)
        ):
            type_issue_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field,
                    "Observed Type": type(value).__name__
                }
            )

    value = row["Value"]

    if (
        value is not None
        and (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
        )
    ):
        type_issue_rows.append(
            {
                "Record Index": int(row_index),
                "Field": "Value",
                "Observed Type": type(value).__name__
            }
        )


type_issues_df = pd.DataFrame(
    type_issue_rows
)

missing_mandatory_df = pd.DataFrame(
    missing_mandatory_rows
)

field_types_valid = type_issues_df.empty
mandatory_fields_complete = missing_mandatory_df.empty

print("Reference schema valid:", reference_schema_valid)
print("Record count valid:", record_count_valid)
print("Category counts valid:", category_counts_valid)
print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)

print(
    json.dumps(
        observed_category_counts,
        ensure_ascii=False,
        indent=2
    )
)

if not reference_schema_valid:
    raise AssertionError(
        "D7 reference schema is invalid."
    )

if not record_count_valid:
    raise AssertionError(
        f"Expected {EXPECTED_REFERENCE_RECORD_COUNT} records, "
        f"found {len(reference_values_df)}."
    )

if not category_counts_valid:
    raise AssertionError(
        "D7 reference category counts are invalid."
    )

if not field_types_valid:
    display(type_issues_df)
    raise AssertionError(
        "D7 reference field types are invalid."
    )

if not mandatory_fields_complete:
    display(missing_mandatory_df)
    raise AssertionError(
        "D7 mandatory reference fields are incomplete."
    )


Reference schema valid: True
Record count valid: True
Category counts valid: True
Field types valid: True
Mandatory fields complete: True
{
  "Education pipeline statistic": 24,
  "Key fact": 10,
  "Government initiative": 9,
  "Policy finding": 7,
  "Recommendation": 6,
  "Policy context": 3
}


In [12]:
# ============================================================
# 11. Duplicate and source-location checks
# ============================================================

complete_duplicate_mask = (
    reference_values_df
    .duplicated(
        subset=EXPECTED_FIELDS,
        keep=False
    )
)

complete_duplicate_count = int(
    complete_duplicate_mask.sum()
)

record_identity_fields = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

duplicate_identity_mask = (
    reference_values_df
    .duplicated(
        subset=record_identity_fields,
        keep=False
    )
)

duplicate_identity_count = int(
    duplicate_identity_mask.sum()
)

source_location_pattern_valid = bool(
    reference_values_df[
        "Source Location"
    ].map(
        lambda value: bool(
            re.match(
                r"^PDF page \d+ — .+$",
                value
            )
        )
    ).all()
)

source_page_numbers = (
    reference_values_df[
        "Source Location"
    ]
    .str.extract(
        r"^PDF page (\d+)"
    )[0]
    .astype(int)
)

source_pages_within_document = bool(
    source_page_numbers.between(
        1,
        EXPECTED_PAGE_COUNT
    ).all()
)

negative_value_count = int(
    reference_values_df[
        "Value"
    ].map(
        lambda value: (
            isinstance(value, (int, float))
            and value < 0
        )
    ).sum()
)

null_numeric_value_count = int(
    reference_values_df[
        "Value"
    ].isna().sum()
)

print("Complete duplicate records:", complete_duplicate_count)
print("Duplicate identity records:", duplicate_identity_count)
print("Source location format valid:", source_location_pattern_valid)
print("Source pages within document:", source_pages_within_document)
print("Negative numeric values:", negative_value_count)
print("Qualitative/null numeric records:", null_numeric_value_count)

qualifier_embedded_in_unit_count = int(
    reference_values_df["Unit"]
    .fillna("")
    .str.contains(
        r"\b(around|almost|over|more than|minimum)\b",
        case=False,
        regex=True
    )
    .sum()
)

print(
    "Qualifiers incorrectly embedded in Unit:",
    qualifier_embedded_in_unit_count
)

if qualifier_embedded_in_unit_count != 0:
    raise AssertionError(
        "One or more qualifiers remain incorrectly embedded "
        "inside the Unit field."
    )

if complete_duplicate_count != 0:
    display(
        reference_values_df.loc[
            complete_duplicate_mask
        ]
    )
    raise AssertionError(
        "Unexpected complete duplicate records detected."
    )

if not source_location_pattern_valid:
    raise AssertionError(
        "One or more source locations have an invalid format."
    )

if not source_pages_within_document:
    raise AssertionError(
        "One or more source locations refer to pages outside "
        "the supplied D7 PDF."
    )


Complete duplicate records: 0
Duplicate identity records: 0
Source location format valid: True
Source pages within document: True
Negative numeric values: 3
Qualitative/null numeric records: 16
Qualifiers incorrectly embedded in Unit: 0


/tmp/ipykernel_5111/1400811879.py:98: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


In [13]:
# ============================================================
# 12. Reference summary and integrity report
# ============================================================

REFERENCE_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "reference_record_count": int(
        len(reference_values_df)
    ),
    "expected_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,
    "record_count_valid":
        record_count_valid,
    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts":
        observed_category_counts,
    "category_counts_valid":
        category_counts_valid,
    "numeric_record_count": int(
        reference_values_df[
            "Value"
        ].notna().sum()
    ),
    "qualitative_record_count":
        null_numeric_value_count,
    "negative_numeric_value_count":
        negative_value_count,
    "source_pages_represented": sorted(
        source_page_numbers.unique().tolist()
    ),
    "fields": EXPECTED_FIELDS
}


REFERENCE_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "source_file": SOURCE_PATH.name,
    "source_file_sha256": SOURCE_SHA256,
    "expected_page_count": EXPECTED_PAGE_COUNT,
    "observed_page_count": PAGE_COUNT,
    "page_count_valid": PAGE_COUNT_VALID,
    "reference_schema_valid":
        reference_schema_valid,
    "record_count_valid":
        record_count_valid,
    "category_counts_valid":
        category_counts_valid,
    "field_types_valid":
        field_types_valid,
    "mandatory_fields_complete":
        mandatory_fields_complete,
    "complete_duplicate_count":
        complete_duplicate_count,
    "duplicate_identity_count":
        duplicate_identity_count,
    "source_location_pattern_valid":
        source_location_pattern_valid,
    "qualifier_embedded_in_unit_count":
        qualifier_embedded_in_unit_count,
    "source_pages_within_document":
        source_pages_within_document,
    "manual_reference_construction":
        True,
    "calculation_applied":
        False,
    "inference_applied":
        False,
    "unit_conversion_applied":
        False,
    "source_value_repair_applied":
        False,
    "reference_integrity_passed": all(
        [
            PAGE_COUNT_VALID,
            reference_schema_valid,
            record_count_valid,
            category_counts_valid,
            field_types_valid,
            mandatory_fields_complete,
            complete_duplicate_count == 0,
            qualifier_embedded_in_unit_count == 0,
            source_location_pattern_valid,
            source_pages_within_document
        ]
    )
}

REFERENCE_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_file_sha256":
        SOURCE_SHA256,

    "reference_file":
        "D7_reference_values.csv",

    "reference_construction_method":
        "Manual document-grounded construction",

    "expected_reference_record_count":
        EXPECTED_REFERENCE_RECORD_COUNT,

    "observed_reference_record_count":
        int(
            len(reference_values_df)
        ),

    "reference_fields":
        EXPECTED_FIELDS,

    "manual_calculation_applied":
        False,

    "semantic_inference_applied":
        False,

    "unit_conversion_applied":
        False,

    "source_value_repair_applied":
        False,

    "reference_values_branch_independent":
        True,

    "reference_values_to_be_reused_for_branches": [
        "A",
        "B",
        "C"
    ],

}

print(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)

print(
    json.dumps(
        REFERENCE_INTEGRITY,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D7",
  "document_name": "UK National Audit Office — Delivering STEM (science, technology, engineering and mathematics) skills for the economy",
  "reference_record_count": 59,
  "expected_record_count": 59,
  "record_count_valid": true,
  "expected_category_counts": {
    "Key fact": 10,
    "Policy context": 3,
    "Policy finding": 7,
    "Education pipeline statistic": 24,
    "Government initiative": 9,
    "Recommendation": 6
  },
  "observed_category_counts": {
    "Education pipeline statistic": 24,
    "Key fact": 10,
    "Government initiative": 9,
    "Policy finding": 7,
    "Recommendation": 6,
    "Policy context": 3
  },
  "category_counts_valid": true,
  "numeric_record_count": 43,
  "qualitative_record_count": 16,
  "negative_numeric_value_count": 3,
  "source_pages_represented": [
    6,
    7,
    8,
    9,
    10,
    11,
    12
  ],
  "fields": [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Value",
    "Unit",
    "

In [14]:
# ============================================================
# 13. Indicator assessment
# ============================================================

indicator_assessment = [
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Reading Order Quality",
        "Score": "Medium",
        "Evidence Source":
            "PDF text extraction + manual document inspection",
        "Justification":
            "The report is predominantly linear, but the Key Facts "
            "page uses a visually distributed layout and relevant "
            "records occur across different summary structures."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Table Structure Integrity",
        "Score": "Medium",
        "Evidence Source":
            "Key Facts layout inspection + extracted text inspection",
        "Justification":
            "The supplied pages do not contain highly complex tables, "
            "but the Key Facts page presents structured values in a "
            "visual facts-panel arrangement that requires preservation "
            "of value-label associations."
    },
    {
        "Dimension": "Structural Readiness",
        "Indicator": "Section/Header Hierarchy",
        "Score": "Low",
        "Evidence Source":
            "Manual report inspection",
        "Justification":
            "The report contains explicit section headings, numbered "
            "summary paragraphs and lettered recommendations that form "
            "a clear document hierarchy."
    },

    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Sharpness",
        "Score": "Low",
        "Evidence Source":
            "Manual visual inspection",
        "Justification":
            "The born-digital PDF is visually clear and all relevant "
            "text is readable."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "Noise / Degradation",
        "Score": "Low",
        "Evidence Source":
            "Manual visual inspection",
        "Justification":
            "No relevant scanning noise, blur or degradation affects "
            "the document."
    },
    {
        "Dimension": "Visual/OCR Readiness",
        "Indicator": "OCR Dependency",
        "Score": "Low",
        "Evidence Source":
            "Automated PDF text extraction",
        "Justification":
            "Text is directly extractable from all 12 pages and OCR "
            "is not required."
    },

    {
        "Dimension": "Semantic Quality",
        "Indicator": "Terminology Consistency",
        "Score": "Medium",
        "Evidence Source":
            "Manual content inspection",
        "Justification":
            "STEM terminology is generally coherent, but the report "
            "explicitly notes that there is no universally accepted "
            "definition of STEM and that government lacks stable and "
            "consistent definitions across contexts."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Schema Alignment",
        "Score": "High",
        "Evidence Source":
            "Reference-schema comparison",
        "Justification":
            "The extraction schema must represent heterogeneous "
            "record types including quantitative facts, policy "
            "context, findings, education statistics, initiatives "
            "and recommendations using a common structure."
    },
    {
        "Dimension": "Semantic Quality",
        "Indicator": "Numerical Density",
        "Score": "Medium",
        "Evidence Source":
            "Automated text profiling + manual inspection",
        "Justification":
            "The report contains numerous quantitative observations, "
            "but the dominant representation remains narrative policy "
            "and audit text rather than dense numerical tables."
    },

    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Required Field Presence",
        "Score": "Low",
        "Evidence Source":
            "Reference-value verification",
        "Justification":
            "All information required by the predefined 59-record "
            "extraction scope is represented in the supplied pages."
    },
    {
        "Dimension": "Completeness and Consistency",
        "Indicator": "Internal Consistency",
        "Score": "Medium",
        "Evidence Source":
            "Manual source and reference inspection",
        "Justification":
            "The report is internally coherent, but several statistics "
            "are repeated across Key Facts and numbered Summary "
            "findings and must remain associated with their respective "
            "source contexts."
    },

    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Format Heterogeneity",
        "Score": "Medium",
        "Evidence Source":
            "Document profiling + manual inspection",
        "Justification":
            "The supplied report mixes publication front matter, a "
            "visual Key Facts panel, numbered narrative findings, "
            "headings and lettered recommendations, but remains "
            "predominantly textual."
    },
    {
        "Dimension":
            "Representation and Normalisation Complexity",
        "Indicator": "Unit / Label Variability",
        "Score": "High",
        "Evidence Source":
            "Reference and source inspection",
        "Justification":
            "The extraction scope contains GBP millions, counts, "
            "percentages, enrolments, apprenticeship starts, teachers, "
            "graduates, qualification levels and explicit qualifiers "
            "such as around, almost, over and more than."
    }
]

indicator_assessment_df = pd.DataFrame(
    indicator_assessment
)

display(indicator_assessment_df)

,Dimension,Indicator,Score,Evidence Source,Justification
0,Structural Readiness,Reading Order Quality,Medium,PDF text extraction + manual document inspection,"The report is predominantly linear, but the Ke..."
1,Structural Readiness,Table Structure Integrity,Medium,Key Facts layout inspection + extracted text i...,The supplied pages do not contain highly compl...
2,Structural Readiness,Section/Header Hierarchy,Low,Manual report inspection,"The report contains explicit section headings,..."
3,Visual/OCR Readiness,Sharpness,Low,Manual visual inspection,The born-digital PDF is visually clear and all...
4,Visual/OCR Readiness,Noise / Degradation,Low,Manual visual inspection,"No relevant scanning noise, blur or degradatio..."
5,Visual/OCR Readiness,OCR Dependency,Low,Automated PDF text extraction,Text is directly extractable from all 12 pages...
6,Semantic Quality,Terminology Consistency,Medium,Manual content inspection,"STEM terminology is generally coherent, but th..."
7,Semantic Quality,Schema Alignment,High,Reference-schema comparison,The extraction schema must represent heterogen...
8,Semantic Quality,Numerical Density,Medium,Automated text profiling + manual inspection,The report contains numerous quantitative obse...
9,Completeness and Consistency,Required Field Presence,Low,Reference-value verification,All information required by the predefined 59-...


In [15]:
# ============================================================
# 14. Validate indicator assessment
# ============================================================

VALID_SCORES = {
    "Low",
    "Medium",
    "High"
}

expected_indicators = {
    "Reading Order Quality",
    "Table Structure Integrity",
    "Section/Header Hierarchy",
    "Sharpness",
    "Noise / Degradation",
    "OCR Dependency",
    "Terminology Consistency",
    "Schema Alignment",
    "Numerical Density",
    "Required Field Presence",
    "Internal Consistency",
    "Format Heterogeneity",
    "Unit / Label Variability"
}

invalid_scores = (
    set(
        indicator_assessment_df[
            "Score"
        ].dropna().unique()
    )
    - VALID_SCORES
)

observed_indicators = set(
    indicator_assessment_df[
        "Indicator"
    ]
)

missing_indicators = (
    expected_indicators
    - observed_indicators
)

unexpected_indicators = (
    observed_indicators
    - expected_indicators
)

if invalid_scores:
    raise ValueError(
        f"Invalid indicator scores: {invalid_scores}"
    )

if missing_indicators:
    raise ValueError(
        f"Missing indicators: {missing_indicators}"
    )

if unexpected_indicators:
    raise ValueError(
        f"Unexpected indicators: {unexpected_indicators}"
    )

if len(indicator_assessment_df) != len(expected_indicators):
    raise ValueError(
        "Duplicate indicator rows detected."
    )

print(
    "Indicator assessment validation passed."
)

Indicator assessment validation passed.


In [16]:
# ============================================================
# 15. Dimension-level assessment
# ============================================================

score_to_numeric = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

indicator_assessment_df[
    "Numeric Score"
] = indicator_assessment_df[
    "Score"
].map(score_to_numeric)


dimension_assessment_df = (
    indicator_assessment_df
    .groupby(
        "Dimension",
        as_index=False
    )
    .agg(
        Mean_Score=(
            "Numeric Score",
            "mean"
        ),
        Number_of_Indicators=(
            "Indicator",
            "count"
        )
    )
)


def classify_dimension_score(mean_score):
    if mean_score < 1.5:
        return "Low"
    elif mean_score < 2.5:
        return "Medium"
    else:
        return "High"


dimension_assessment_df[
    "Dimension Score"
] = dimension_assessment_df[
    "Mean_Score"
].apply(
    classify_dimension_score
)

dimension_assessment_df[
    "Mean_Score"
] = dimension_assessment_df[
    "Mean_Score"
].round(2)

display(
    dimension_assessment_df
)

,Dimension,Mean_Score,Number_of_Indicators,Dimension Score
0,Completeness and Consistency,1.50,2,Medium
1,Representation and Normalisation Complexity,2.50,2,High
2,Semantic Quality,2.33,3,Medium
3,Structural Readiness,1.67,3,Medium
4,Visual/OCR Readiness,1.00,3,Low


In [17]:
# ============================================================
# 16. Structured quality-assessment evidence
# ============================================================

QUALITY_EVIDENCE = {
    "document_id":
        DOCUMENT_ID,

    "assessment_basis":
        "Observed document evidence was mapped to the "
        "predefined Low, Medium, and High operational "
        "criteria defined in Table 3.3 of the methodology.",

    "evidence_method":
        "Evidence was obtained through automated profiling "
        "where measurable characteristics could be derived "
        "programmatically and through documented manual "
        "inspection where qualitative assessment was required.",

    "indicators":
        indicator_assessment_df[
            [
                "Dimension",
                "Indicator",
                "Score",
                "Evidence Source",
                "Justification"
            ]
        ].to_dict(
            orient="records"
        ),

    "dimension_aggregation": {
        "encoding": {
            "Low": 1,
            "Medium": 2,
            "High": 3
        },

        "aggregation":
            "Arithmetic mean of indicator scores within "
            "each dimension.",

        "classification_rule": {
            "Low":
                "mean < 1.5",

            "Medium":
                "1.5 <= mean < 2.5",

            "High":
                "mean >= 2.5"
        }
    },

    "dimensions":
        dimension_assessment_df[
            [
                "Dimension",
                "Mean_Score",
                "Dimension Score"
            ]
        ].to_dict(
            orient="records"
        )
}

In [18]:
# ============================================================
# 17. Export Stage 1 outputs
# ============================================================

reference_values_df.to_csv(
    REFERENCE_VALUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

REFERENCE_VALUES_JSON_PATH.write_text(
    json.dumps(
        reference_records,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_SCHEMA_PATH.write_text(
    json.dumps(
        REFERENCE_SCHEMA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_SUMMARY_PATH.write_text(
    json.dumps(
        REFERENCE_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

DOCUMENT_CHARACTERISATION_PATH.write_text(
    json.dumps(
        DOCUMENT_CHARACTERISATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

QUALITY_EVIDENCE_PATH.write_text(
    json.dumps(
        QUALITY_EVIDENCE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_INTEGRITY_PATH.write_text(
    json.dumps(
        REFERENCE_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

page_characterisation_df.to_csv(
    PAGE_CHARACTERISATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

indicator_assessment_df[
    [
        "Dimension",
        "Indicator",
        "Score",
        "Evidence Source",
        "Justification"
    ]
].to_csv(
    INDICATOR_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)

dimension_assessment_df.to_csv(
    DIMENSION_ASSESSMENT_PATH,
    index=False,
    encoding="utf-8-sig"
)

EXTRACTION_SCHEMA_PATH.write_text(
    json.dumps(
        EXTRACTION_SCHEMA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

EXTRACTION_TASK_PATH.write_text(
    EXTRACTION_TASK.strip(),
    encoding="utf-8",
    newline="\n"
)

REFERENCE_METADATA_PATH.write_text(
    json.dumps(
        REFERENCE_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8",
    newline="\n"
)

print("D7 Stage 1 outputs exported.")


D7 Stage 1 outputs exported.


In [19]:
# ============================================================
# 18. Final checks and outputs listing
# ============================================================

GENERATED_OUTPUTS = [
    REFERENCE_VALUES_PATH,
    REFERENCE_VALUES_JSON_PATH,
    REFERENCE_SCHEMA_PATH,
    EXTRACTION_SCHEMA_PATH,
    EXTRACTION_TASK_PATH,
    REFERENCE_SUMMARY_PATH,
    REFERENCE_METADATA_PATH,
    DOCUMENT_CHARACTERISATION_PATH,
    PAGE_CHARACTERISATION_PATH,
    INDICATOR_ASSESSMENT_PATH,
    DIMENSION_ASSESSMENT_PATH,
    QUALITY_EVIDENCE_PATH,
    REFERENCE_INTEGRITY_PATH
]

missing_outputs = [
    path.name
    for path in GENERATED_OUTPUTS
    if not path.exists()
]

if missing_outputs:
    raise AssertionError(
        f"Missing D7 Stage 1 outputs: {missing_outputs}"
    )

if not REFERENCE_INTEGRITY[
    "reference_integrity_passed"
]:
    raise AssertionError(
        "D7 reference integrity checks failed."
    )

print("D7 Stage 1 completed successfully.")
print("Generated files:")

for path in GENERATED_OUTPUTS:
    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )


D7 Stage 1 completed successfully.
Generated files:
- D7_reference_values.csv | exists: True
- D7_reference_values.json | exists: True
- D7_reference_schema.json | exists: True
- D7_extraction_schema.json | exists: True
- D7_extraction_task.txt | exists: True
- D7_reference_summary.json | exists: True
- D7_reference_metadata.json | exists: True
- D7_document_characterisation.json | exists: True
- D7_page_characterisation.csv | exists: True
- D7_indicator_assessment.csv | exists: True
- D7_dimension_assessment.csv | exists: True
- D7_quality_evidence.json | exists: True
- D7_reference_integrity.json | exists: True
